# Fine-Tuning **Versi 2** Qwen 2.5 3B Instruct — Chatbot Regulasi Kampus (QLoRA + Unsloth)

**Beda dengan versi 1:** dataset diperbanyak dengan 3 variasi pertanyaan per fakta (±2.300 contoh, dari `parafrase_v2_colab.ipynb`), test set berisi variasi kalimat baru dari fakta yang sudah diajarkan, LoRA rank 16 → 32, batch efektif 8 → 16. Tujuannya agar model **hafal fakta**, bukan hanya meniru gaya jawaban.

Notebook ini dijalankan di **Google Colab (GPU T4 gratis)**. Hasil akhirnya adalah file **GGUF** yang di-download lalu di-deploy **lokal di laptop memakai Ollama**.

**Sebelum mulai:**
1. `Runtime → Change runtime type → T4 GPU`.
2. Pastikan `train_v2.jsonl`, `val_v2.jsonl`, `test_v2.jsonl` (hasil `parafrase_v2_colab.ipynb`) ke Google Drive folder `MyDrive/proyek-llm/data/processed/`.
3. Jalankan sel dari atas ke bawah. Semua output (log, kurva loss, adapter LoRA, GGUF) disimpan ke `MyDrive/proyek-llm/outputs/` supaya tidak hilang kalau sesi Colab terputus.

## 1. Instalasi library

In [ ]:
%%capture
!pip install unsloth

## 2. Mount Google Drive & konfigurasi

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
BASE_DIR = '/content/drive/MyDrive/proyek-llm'
DATA_DIR = f'{BASE_DIR}/data/processed'
OUT_DIR  = f'{BASE_DIR}/outputs_v2'
os.makedirs(OUT_DIR, exist_ok=True)
print(os.listdir(DATA_DIR))

In [ ]:
# HARUS sama persis dengan scripts/config.py dan deploy/Modelfile
NAMA_KAMPUS = "Universitas Telkom"
SYSTEM_PROMPT = (
    f"Kamu adalah asisten layanan informasi akademik {NAMA_KAMPUS}. "
    "Jawab pertanyaan mahasiswa tentang peraturan dan regulasi akademik kampus "
    "dengan bahasa Indonesia yang sopan, singkat, dan jelas. "
    "Jika informasinya tidak ada dalam peraturan, katakan bahwa kamu tidak tahu "
    "dan sarankan menghubungi bagian akademik."
)

MODEL_NAME     = "unsloth/Qwen2.5-3B-Instruct"
MAX_SEQ_LENGTH = 2048

# Hyperparameter LoRA / training (jelaskan alasannya di laporan)
LORA_R       = 32     # v2: rank dinaikkan (v1 = 16) agar kapasitas menghafal fakta lebih besar
LORA_ALPHA   = 32     # alpha = r
LORA_DROPOUT = 0
EPOCHS       = 3      # tiap fakta kini muncul ±4 variasi x 3 epoch = ±12 kali (v1: 3 kali)
LR           = 2e-4
BATCH_SIZE   = 4
GRAD_ACCUM   = 4      # effective batch = 16

## 3. Load base model (4-bit / QLoRA)

In [ ]:
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = MODEL_NAME,
    max_seq_length = MAX_SEQ_LENGTH,
    dtype          = None,      # auto: float16 di T4
    load_in_4bit   = True,      # QLoRA
)

## 4. Pasang adapter LoRA

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = LORA_R,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = LORA_ALPHA,
    lora_dropout = LORA_DROPOUT,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)
model.print_trainable_parameters()

## 5. Load & format dataset (chat template Qwen 2.5 / ChatML)

In [ ]:
from datasets import load_dataset
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(tokenizer, chat_template="qwen-2.5")

ds = load_dataset("json", data_files={
    "train": f"{DATA_DIR}/train_v2.jsonl",
    "val":   f"{DATA_DIR}/val_v2.jsonl",
    "test":  f"{DATA_DIR}/test_v2.jsonl",
})

def format_chat(batch):
    return {"text": [tokenizer.apply_chat_template(m, tokenize=False, add_generation_prompt=False)
                     for m in batch["messages"]]}

# Sisakan hanya kolom "text" agar SFTTrainer tidak memproses ulang kolom "messages"
train_ds = ds["train"].map(format_chat, batched=True, remove_columns=ds["train"].column_names)
val_ds   = ds["val"].map(format_chat, batched=True, remove_columns=ds["val"].column_names)
print(ds)
print(train_ds[0]["text"])

In [ ]:
# Cek panjang token -> pastikan tidak ada yang melebihi MAX_SEQ_LENGTH
lens = [len(tokenizer(t)["input_ids"]) for t in train_ds["text"]]
print(f"token min={min(lens)}, rata2={sum(lens)/len(lens):.0f}, maks={max(lens)}")

## 6. Jawaban model SEBELUM fine-tuning (baseline)
Adapter LoRA masih nol, jadi output di sini = model dasar.

In [ ]:
import pandas as pd

def jawab(pertanyaan, max_new_tokens=256):
    msgs = [{"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": pertanyaan}]
    enc = tokenizer.apply_chat_template(msgs, add_generation_prompt=True, return_dict=True,
                                        return_tensors="pt").to("cuda")
    out = model.generate(**enc, max_new_tokens=max_new_tokens,
                         do_sample=False, use_cache=True)
    n_in = enc["input_ids"].shape[1]
    return tokenizer.decode(out[0][n_in:], skip_special_tokens=True).strip()

test_q   = [m[1]["content"] for m in ds["test"]["messages"]]   # semua soal test (±72)
test_ref = [m[2]["content"] for m in ds["test"]["messages"]]

FastLanguageModel.for_inference(model)
sebelum = [jawab(q) for q in test_q]
for q, a in list(zip(test_q, sebelum))[:3]:
    print("Q:", q, "\nA:", a, "\n")

## 7. Training

In [ ]:
from trl import SFTTrainer, SFTConfig
from unsloth import is_bfloat16_supported
from unsloth.chat_templates import train_on_responses_only

FastLanguageModel.for_training(model)

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_ds,
    eval_dataset = val_ds,
    args = SFTConfig(
        dataset_text_field = "text",
        per_device_train_batch_size = BATCH_SIZE,
        gradient_accumulation_steps = GRAD_ACCUM,
        num_train_epochs = EPOCHS,
        learning_rate = LR,
        warmup_ratio = 0.05,
        lr_scheduler_type = "linear",
        optim = "adamw_8bit",
        weight_decay = 0.01,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 10,
        eval_strategy = "steps",
        eval_steps = 50,
        save_strategy = "no",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none",
    ),
)

# Hitung loss hanya pada bagian jawaban asisten (bukan system/pertanyaan)
trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|im_start|>user\n",
    response_part    = "<|im_start|>assistant\n",
)

In [ ]:
gpu = torch.cuda.get_device_properties(0)
print(f"GPU: {gpu.name}, VRAM: {gpu.total_memory/1024**3:.1f} GB")

stats = trainer.train()

print(f"Waktu training: {stats.metrics['train_runtime']/60:.1f} menit")
print(f"VRAM puncak: {torch.cuda.max_memory_reserved()/1024**3:.2f} GB")

## 8. Log & kurva loss (lampirkan di laporan)

In [ ]:
import matplotlib.pyplot as plt

log = pd.DataFrame(trainer.state.log_history)
log.to_csv(f"{OUT_DIR}/training_log.csv", index=False)

tr = log.dropna(subset=["loss"])
ev = log.dropna(subset=["eval_loss"])
plt.figure(figsize=(8, 4.5))
plt.plot(tr["step"], tr["loss"], label="train loss")
plt.plot(ev["step"], ev["eval_loss"], "o-", label="validation loss")
plt.xlabel("step"); plt.ylabel("loss"); plt.title("Kurva Loss Fine-Tuning")
plt.legend(); plt.grid(alpha=.3)
plt.savefig(f"{OUT_DIR}/loss_curve.png", dpi=150, bbox_inches="tight")
plt.show()
log.tail(10)

## 9. Jawaban SESUDAH fine-tuning & perbandingan

In [ ]:
FastLanguageModel.for_inference(model)
sesudah = [jawab(q) for q in test_q]

banding = pd.DataFrame({"pertanyaan": test_q, "jawaban_referensi": test_ref,
                        "sebelum_finetune": sebelum, "sesudah_finetune": sesudah})
banding.to_csv(f"{OUT_DIR}/perbandingan_sebelum_sesudah_v2.csv", index=False)
pd.set_option("display.max_colwidth", 200)
banding.head(10)

In [ ]:
# Skor sederhana: token F1 terhadap jawaban referensi + cek angka penting
import re
from collections import Counter
def kata(s): return re.findall(r"\w+", s.lower())
def f1(p, r):
    a, b = Counter(kata(p)), Counter(kata(r)); s = sum((a & b).values())
    return 0 if s == 0 else 2 * (s / sum(a.values())) * (s / sum(b.values())) / (s / sum(a.values()) + s / sum(b.values()))
def angka_cocok(p, r):
    ar = set(re.findall(r"\d+(?:[.,]\d+)?", r))
    return None if not ar else ar <= set(re.findall(r"\d+(?:[.,]\d+)?", p))
banding["f1_sebelum"] = [f1(p, r) for p, r in zip(sebelum, test_ref)]
banding["f1_sesudah"] = [f1(p, r) for p, r in zip(sesudah, test_ref)]
banding["angka_benar_sesudah"] = [angka_cocok(p, r) for p, r in zip(sesudah, test_ref)]
ada_angka = banding["angka_benar_sesudah"].dropna()
print(f"Token F1  sebelum: {banding.f1_sebelum.mean():.3f}   sesudah: {banding.f1_sesudah.mean():.3f}")
print(f"Soal berangka yang angkanya benar (sesudah): {int(ada_angka.sum())}/{len(ada_angka)}")
banding.to_csv(f"{OUT_DIR}/perbandingan_sebelum_sesudah_v2.csv", index=False)

In [ ]:
# Coba pertanyaan bebas
print(jawab("Kalau IPK saya di bawah 2, apa yang terjadi?"))

## 10. Simpan adapter LoRA & export GGUF (Q4_K_M)

In [ ]:
model.save_pretrained(f"{OUT_DIR}/lora_adapter")
tokenizer.save_pretrained(f"{OUT_DIR}/lora_adapter")

# Proses ini ~10-20 menit (merge + konversi + kuantisasi)
QUANT = "q4_k_m"   # opsi lebih akurat: "q8_0" (±3,3 GB, tetap kuat dijalankan di laptop 16 GB)
model.save_pretrained_gguf("chatbot_kampus_v2_gguf", tokenizer, quantization_method=QUANT)

In [ ]:
import glob, shutil
ggufs = [g for g in glob.glob("/content/**/*.gguf", recursive=True) if "drive" not in g]
print(ggufs)
for g in ggufs:
    if QUANT in g.lower():
        tujuan = f"{OUT_DIR}/chatbot-kampus-v2-{QUANT}.gguf"
        shutil.copy(g, tujuan)
        print("Tersalin ke Drive:", tujuan)

## 11. Langkah berikutnya (di laptop)
1. Download `outputs_v2/chatbot-kampus-v2-q4_k_m.gguf` ke `D:\proyek-llm\deploy\`.
2. `cd D:\proyek-llm\deploy` lalu `ollama create chatbot-kampus-v2 -f Modelfile.v2`.
3. Ubah `FINETUNED_MODEL = "chatbot-kampus-v2"` di `scripts/config.py`, lalu `streamlit run deploy/app.py`.
4. Download `loss_curve.png`, `training_log.csv`, `perbandingan_sebelum_sesudah_v2.csv` dari `outputs_v2/` ke `docs/v2/`.